In [1]:
!git clone https://github.com/GyanRout/Major-Projects.git
%cd Major-Projects
!git pull

Cloning into 'Major-Projects'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 69 (delta 19), reused 43 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (69/69), 1.70 MiB | 6.36 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/Major-Projects
Already up to date.


In [2]:
!pip install -r /content/Major-Projects/dp_recommender_project/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.8 MB/s eta 0:00:00


In [3]:
import sys

# Adding project root directory to Python's path
project_root = '/content/Major-Projects/dp_recommender_project'
if project_root not in sys.path:
    sys.path.append(project_root)

In [4]:
from src.model_defs import RecommenderModel
from src.data_utils import Loader
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error
import numpy as np
import json
import pandas as pd
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

user_idx_path = '/content/Major-Projects/dp_recommender_project/data/user_to_index.json'
movie_idx_path = '/content/Major-Projects/dp_recommender_project/data/movie_to_index.json'
test_data_path = '/content/Major-Projects/dp_recommender_project/data/test_data.csv'
load_path = '/content/Major-Projects/dp_recommender_project/src/private_recommender.pth'

with open(user_idx_path,'r') as f:
  NUM_USER = len(json.load(f))

with open(movie_idx_path,'r') as f:
  NUM_ITEM = len(json.load(f))

test_df = pd.read_csv(test_data_path)
test_dataset = Loader(test_df)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

model = RecommenderModel(num_user=NUM_USER, num_item=NUM_ITEM)

loaded_state_dict = torch.load(load_path, map_location=device, weights_only=True)
model.load_state_dict(loaded_state_dict)
model = model.to(device)

In [5]:
def evalute_rmse(model, dataloader, device):
  model.eval()

  all_prediction=[]
  all_target=[]

  with torch.inference_mode():
    for users, items, rating in dataloader:
      users = users.to(device)
      items = items.to(device)

      pred = model(users, items)

      all_prediction.extend(pred.squeeze().cpu().numpy())
      all_target.extend(rating.numpy())

  mse = mean_squared_error(all_target, all_prediction)
  rmse = np.sqrt(mse)

  return rmse

RMSE = evalute_rmse(model, test_loader, device)

print(f"The RMSE of our private model is {RMSE:.4f}")

The RMSE of our private model is 0.9564


In [10]:
# For Comaprison and Testing Purpose only
# Creating a Recommender Model without our privacy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import json

# Ensure you have imported your custom modules correctly based on your path setup
from src.data_utils import Loader
from src.model_defs import RecommenderModel

# --- 1. System & Hyperparameter Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 15
BATCH_SIZE = 2048  # Massive batch size is safe here because Autograd only tracks aggregate gradients
LEARNING_RATE = 0.01

# Paths (Adjust these if your Colab directory differs)
base_path = '/content/Major-Projects/dp_recommender_project'
train_data_path = f'{base_path}/data/train_data.csv'
user_idx_path = f'{base_path}/data/user_to_index.json'
movie_idx_path = f'{base_path}/data/movie_to_index.json'
save_path = f'{base_path}/src/standard_recommender.pth'

# --- 2. Data Ingestion ---
with open(user_idx_path, 'r') as f:
    NUM_USERS = len(json.load(f))
with open(movie_idx_path, 'r') as f:
    NUM_ITEMS = len(json.load(f))

train_df = pd.read_csv(train_data_path)
train_dataset = Loader(train_df)

# Standard DataLoader (No micro-batching required)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

# --- 3. Model & Optimizer Instantiation ---
# Instantiate the exact same blueprint
standard_model = RecommenderModel(num_user=NUM_USERS, num_item=NUM_ITEMS)
standard_model = standard_model.to(device)

optimizer = optim.Adam(standard_model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.HuberLoss()

# --- 4. The Standard Training Loop ---
print("Initiating Standard Training (No Privacy Constraints)...")
standard_model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    batch_count = 0

    for user_ids, item_ids, ratings in train_loader:
        # Move data to GPU
        user_ids = user_ids.to(device)
        item_ids = item_ids.to(device)
        ratings = ratings.to(device).view(-1, 1)

        # Standard PyTorch Execution
        optimizer.zero_grad()
        predictions = standard_model(user_ids, item_ids)
        loss = loss_fn(predictions, ratings)

        loss.backward()
        optimizer.step()

        # Detach loss to prevent VRAM memory leak
        epoch_loss += loss.item()
        batch_count += 1

    avg_loss = epoch_loss / batch_count
    print(f"Epoch: {epoch+1}/{EPOCHS} | Average Huber Loss: {avg_loss:.4f}")

# --- 5. Serialization (Saving the Parameters) ---
# Because there is no Opacus wrapper, we do not need to use ._module
# We can save the state_dict directly.
torch.save(standard_model.state_dict(), save_path)
print(f"\nStandard baseline model parameters saved successfully to:\n{save_path}")

Initiating Standard Training (No Privacy Constraints)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch: 1/15 | Average Huber Loss: 0.4708
Epoch: 2/15 | Average Huber Loss: 0.3036
Epoch: 3/15 | Average Huber Loss: 0.2016
Epoch: 4/15 | Average Huber Loss: 0.1408
Epoch: 5/15 | Average Huber Loss: 0.1067
Epoch: 6/15 | Average Huber Loss: 0.0864
Epoch: 7/15 | Average Huber Loss: 0.0735
Epoch: 8/15 | Average Huber Loss: 0.0644
Epoch: 9/15 | Average Huber Loss: 0.0577
Epoch: 10/15 | Average Huber Loss: 0.0525
Epoch: 11/15 | Average Huber Loss: 0.0488
Epoch: 12/15 | Average Huber Loss: 0.0458
Epoch: 13/15 | Average Huber Loss: 0.0429
Epoch: 14/15 | Average Huber Loss: 0.0410
Epoch: 15/15 | Average Huber Loss: 0.0393

Standard baseline model parameters saved successfully to:
/content/Major-Projects/dp_recommender_project/src/standard_recommender.pth


In [11]:
standard_model = RecommenderModel(num_user=NUM_USER, num_item=NUM_ITEM)

load_path_no_privacy = '/content/Major-Projects/dp_recommender_project/src/standard_recommender.pth'

loaded_state_dict = torch.load(load_path_no_privacy, map_location=device, weights_only=True)
standard_model.load_state_dict(loaded_state_dict)
standard_model = standard_model.to(device)

standard_model_RMSE = evalute_rmse(standard_model, test_loader, device)
print(f"The RMSE of standard model is {standard_model_RMSE:.4f}")
print(f"We traded {RMSE-standard_model_RMSE:.4f} stars for privacy")

The RMSE of standard model is 1.0322
We traded -0.0758 stars for privacy
